# NB14-T6 — Posterior caches for the stage-2 checkpoints (SPEC T6, GPU part)

Runs each kept checkpoint **once** and saves raw float32 frame logits, greedy decodes and emission spans
(decision 26). Only what T6 uses is cached (SPEC compute budget):

| Checkpoints | Splits |
| --- | --- |
| run A × 3 | fold B (verifier training, out-of-fold), Extra-dev, Iqra dev slice |
| run B × 3 | fold A (verifier training, out-of-fold), Extra-dev, Iqra dev slice |
| BASE, best checkpoint | Extra-dev (ladder row 1 with a bootstrap interval) |

Uses both GPUs when the session has two (one checkpoint per GPU at a time). All analysis (ensemble, verifiers,
LM, ladder) runs in **NB15-T6 on CPU**, so it costs no GPU quota. Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator **GPU T4 x2** (one T4 also works, about twice as long), Internet *On*.
Add data: `iqra-cache`, the NB11 output, the NB8 model output (notebook `notebook17aed3d165`, parity check only),
and the three NB13 outputs: **A** (`iqra-stage2-run-a`), **B** (notebook `notebook7abb4f4be3`, latest version)
and **BASE**: attached automatically from version 5 of that notebook with `kagglehub` (no dataset needed).
If both A and B came from the same notebook, attach B as the dataset `iqra-stage2-run-b` instead.
Runs are recognised by their `run_summary.json`, not by name. Secrets: `HF_TOKEN`.

**After the run:** Save Version. NB15-T6 attaches this notebook's output. Paste back the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Environment, parity (decision 27), inputs, runs, cache plan -----------------------------
import json, queue, threading, time
import numpy as np
import pandas as pd
import torch, transformers
from transformers import AutoFeatureExtractor
from datasets import load_dataset, load_from_disk
from iqra import data, ensemble as E, model as M, parity, posteriors, scorer
from iqra.config import Config, find_cache_root, find_input
from iqra.decode import greedy_decode
from iqra.vocab import Vocab

DEVICES = [f"cuda:{i}" for i in range(min(torch.cuda.device_count(), 2))] or ["cpu"]
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e
T_START = time.time()
cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
OUT = "/kaggle/working/nb14_t6"
os.makedirs(OUT, exist_ok=True)
vocab = Vocab.load(cfg.paths.vocab)
ID2TOK, BLANK = vocab.id2tok, vocab.blank_id
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)

# Find the runs first: a missing input stops here in seconds, before any model is loaded.
# BASE ran as version 5 of the training notebook, whose latest version (10) is run B; attaching
# the notebook gives only its latest version, so BASE is attached by version with kagglehub
# (inside Kaggle this mounts the output, no download) unless a BASE output is already attached.
BASE_SOURCE = "qasimmaajid/notebook7abb4f4be3/versions/5"
ROOTS = ["/kaggle/input"]
if "BASE" not in E.find_runs(ROOTS, required=()):
    import kagglehub
    ROOTS.append(kagglehub.notebook_output_download(BASE_SOURCE))
    print("BASE attached from", BASE_SOURCE, "->", ROOTS[-1])
runs = E.find_runs(ROOTS)                  # by run_summary.json content; checks A, B, BASE all present
plan = E.cache_plan(runs)                  # checks each run trained on the folds the design says
E.assert_out_of_fold(plan, runs, splits)   # rule 6 / SPEC risk table
for r in E.EXPECTED_RUNS:
    print(f"run {r:4s} folds {runs[r]['extra_folds']} | kept {[(m.step, m.f1) for m in runs[r]['members']]} | {runs[r]['dir']}")
for m, s in plan:
    print(f"  cache {m.name:10s} on {s}")
par = parity.verify_environment(DEVICES[0])     # decision 27
print("version parity: PASS |", par["here"], "| devices:", DEVICES,
      [torch.cuda.get_device_name(i) for i in range(len(DEVICES))] if DEVICES[0] != "cpu" else "")
fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")      # as NB2 / NB8 / NB13
report = {"repo_commit": REPO_COMMIT, "versions": par["here"], "parity": par["pass"], "devices": DEVICES,
          "members": E.summarise_members(runs), "plan": [[m.name, s] for m, s in plan]}


In [ ]:
# ---- Audio for the four splits, decoded once into memory ---------------------------------------
extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)
ex_ids = [str(x) for x in list(extra["id"])]
assert ex_ids == splits["extra"]["ids_by_row"], "Iqra_Extra_IS26 row order changed since NB11"
row_of = {c: r for r, c in enumerate(ex_ids)}
dev = load_from_disk(cfg.paths.train_arrow)["dev"]
slice_ids, slice_rows = splits["iqra_dev_slice"]["ids"], splits["iqra_dev_slice"]["rows"]
assert [str(dev["id"][r]) for r in slice_rows[:5]] == slice_ids[:5], "Iqra dev slice changed"

def audio_of(ds, row):
    return data.to_16k(*data.decode_audio(ds.data.slice(row, 1).column("audio").to_pylist()[0]))

AUDIO = {s: {c: audio_of(extra, row_of[c]) for c in splits["extra"][s]} for s in ("extra_dev", "fold_a", "fold_b")}
AUDIO["iqra_dev_slice"] = {c: audio_of(dev, r) for c, r in zip(slice_ids, slice_rows)}
assert set(AUDIO["extra_dev"]) == set(truth_dev["ID"])
hours = {s: round(sum(len(a) for a in v.values()) / 16_000 / 3600, 3) for s, v in AUDIO.items()}
print({s: len(v) for s, v in AUDIO.items()}, "| hours", hours)
work_h = sum(hours[s] for _, ss in plan for s in ss)
print(f"total audio to run: {work_h:.2f} h over {len(plan)} checkpoints on {len(DEVICES)} device(s)")


In [ ]:
# ---- Run every checkpoint once per planned split (one worker per GPU) ---------------------------
def cache_member(member, split_names, device):
    model, info = M.load_model(member.path, device)
    assert not any(info.values()), f"{member.name} did not load cleanly: {info}"
    assert model.config.vocab_size == len(vocab.tok2id) and model.config.pad_token_id == BLANK
    rate = posteriors.model_frame_rate(model.config)
    stats = {}
    for s in split_names:
        path = f"{OUT}/{E.cache_name(member.name, s)}"
        if os.path.exists(path):                      # re-run in the same session: keep what is done
            stats[s] = "already cached"
            continue
        t0, entries = time.time(), {}
        for cid, a in AUDIO[s].items():
            entries[cid] = posteriors.make_entry(posteriors.clip_logits(model, fe, a, device), ID2TOK, BLANK)
        posteriors.write_cache(path, entries, checkpoint=member.name, split=s, vocab_hash=vocab.hash, frame_rate=rate,
                               extra_meta={"repo_commit": REPO_COMMIT, "run": member.run, "step": member.step,
                                           "t5_f1": member.f1, "device": device})
        stats[s] = {"clips": len(entries), "minutes": round((time.time() - t0) / 60, 2),
                    "mb": round(os.path.getsize(path) / 1e6, 1)}
    del model
    if device.startswith("cuda"):
        torch.cuda.empty_cache()
    return stats

jobs, lock, errors, cache_stats = queue.Queue(), threading.Lock(), [], {}
for item in plan:
    jobs.put(item)
t_cache = time.time()

def worker(device):
    while True:
        try:
            member, split_names = jobs.get_nowait()
        except queue.Empty:
            return
        try:
            t0 = time.time()
            st = cache_member(member, split_names, device)
            with lock:
                cache_stats[member.name] = st
                done = len(cache_stats)
                el = (time.time() - t_cache) / 60
                print(f"[{device}] {member.name}: {st} | {(time.time() - t0) / 60:.1f} min | "
                      f"{done}/{len(plan)} done, {el:.1f} min elapsed, ~{el / done * (len(plan) - done):.0f} min left")
        except Exception as e:
            with lock:
                errors.append((member.name, repr(e)))
                print(f"[{device}] {member.name} FAILED: {e!r}")

threads = [threading.Thread(target=worker, args=(d,)) for d in DEVICES]
for t in threads:
    t.start()
for t in threads:
    t.join()
assert not errors, errors
report["caches"] = cache_stats
report["cache_minutes"] = round((time.time() - t_cache) / 60, 1)
print(f"all caches written in {report['cache_minutes']} min")


In [ ]:
# ---- Checks: each cache reproduces its T5 Extra-dev F1; frames agree; ensemble preview ----------
loaded = {}
for m, ss in plan:
    for s in ss:
        meta, clips = posteriors.read_cache(f"{OUT}/{E.cache_name(m.name, s)}")
        assert meta["vocab_hash"] == vocab.hash and set(clips) == set(AUDIO[s]), (m.name, s)
        loaded.setdefault(m.name, {})[s] = clips
posteriors.check_compatible([posteriors.read_cache(f"{OUT}/{E.cache_name(m.name, 'extra_dev')}")[0] for m, _ in plan])

def pred_of(entries):
    return pd.DataFrame({"ID": list(entries), "Prediction": [" ".join(e["tokens"]) for e in entries.values()]})

repro = []
for m, _ in plan:
    f1 = scorer.official_score(truth_dev, pred_of(loaded[m.name]["extra_dev"]))["f1"]
    repro.append({"member": m.name, "t5_f1": m.f1, "cache_f1": f1, "diff": round(f1 - m.f1, 4)})
repro = pd.DataFrame(repro)
print(repro.to_string(index=False))
# Same checkpoint, same clips, single-clip float32 as in T5's evaluator: the scores should match.
assert (repro["diff"].abs() <= 0.005).all(), "a cache does not reproduce its checkpoint's T5 score"
report["reproduce_t5_f1"] = repro.to_dict(orient="records")

members6 = [m.name for m, _ in plan if m.run in ("A", "B")]
for s in ("extra_dev", "iqra_dev_slice"):
    bad = E.frames_agree({n: loaded[n][s] for n in members6}, members6)
    assert not bad, (s, bad[:3])
ens = E.ensemble_split({n: loaded[n]["extra_dev"] for n in members6}, members6, ID2TOK, BLANK)
prev = scorer.official_score(truth_dev, pred_of(ens))
print(f"preview, 6-checkpoint ensemble on Extra-dev (full analysis in NB15-T6): "
      f"F1 {prev['f1']} P {prev['precision']} R {prev['recall']}")
report["ensemble_preview"] = {k: prev[k] for k in ("precision", "recall", "f1")}
report["total_minutes"] = round((time.time() - T_START) / 60, 1)
with open(f"{OUT}/nb14_t6_report.json", "w") as f:
    json.dump(report, f, indent=1, default=str)


In [ ]:
# ---- PASTE BACK -------------------------------------------------------------------------------
print(json.dumps(report, indent=1, default=str))
print("files:", sorted(os.listdir(OUT)))
